# Module 3: User Interface
### BWSI SGAI 2026

By the end of this module you'll know:
- Where each UI component lives in the `ui/` directory
- How to change the game's colors with one dict in `constants.py`
- How to toggle between analog and digital clock modes with one flag
- How the game window is assembled from six components

---
*Open this notebook in JupyterLab. Press `Alt+R` to enter slideshow mode.*  
*Students: just scroll through normally and run the exercise cells.*

## What Is the UI?

The game window is built from **six separate components** assembled together:

| Component | File | What it does |
|-----------|------|--------------|
| `UI` | `ui/ui.py` | Root window — creates and wires all components |
| `GameViewer` | `ui/game_viewer.py` | Central image panel — shows the current humanoid |
| `ButtonMenu` | `ui/button_menu.py` | Four action buttons (Save, Skip, Squish, Scram) |
| `Clock` | `ui/clock.py` | Analog clock (default) |
| `DigitalClock` | `ui/digital_clock.py` | Digital countdown (alternative) |
| `CapacityMeter` | `ui/capacity_meter.py` | Ambulance slot grid |

The rendering library is **Tkinter** — it ships with Python, no install needed.

> Note: Tkinter opens a real OS window with its own event loop. You cannot instantiate UI classes inside a Jupyter notebook cell — they need a display and will block the kernel. We'll inspect them with `inspect` and `import` instead.

## Where does this live in the code?

```
bwsi-sgai/
├── gameplay/
├── models/
├── game_interfaces/
├── ui/                  <- this module
│   ├── ui.py           <- now
│   ├── button_menu.py  <- now
│   ├── game_viewer.py  <- now
│   ├── clock.py        <- now
│   ├── digital_clock.py <- now
│   ├── capacity_meter.py <- now
│   └── graphics/        <- images, clock face
├── training/
└── tests/
```

*The `ui/` directory owns everything you see on screen.*  
*Colors and clock style are controlled in `gameplay/constants.py` using feature flags, not here.*

*`bwsi-sgai/ui/ui.py`*

## How the Window Is Assembled — Locate

Open `ui/ui.py`. The `UI` class is the top-level container:

```
UI
├── root (tk.Tk)          root window, title, background color
├── _create_header()      game logo + BWSI logo bar across the top
├── ButtonMenu            action buttons placed left-side
├── GameViewer            humanoid image canvas (center)
├── Clock or DigitalClock timer widget (top-right) — chosen by LEGACY_CLOCK
├── CapacityMeter         ambulance slot grid (right)
└── value_label           humanoid priority score label
```

**`UI` is the entry point and the wiring harness.**  
Each sub-component handles its own rendering; `UI` coordinates them.

*`bwsi-sgai/ui/ui.py`*

## How the Window Is Assembled — Orient

The feature-flag pattern at the top of `ui.py` is worth reading closely:

```python
# Feature flag: swap clock implementations by changing LEGACY_CLOCK in gameplay/constants.py.
# This is a teaching example of the feature-flag pattern.
_ClockClass = Clock if LEGACY_CLOCK else DigitalClock
```

Then inside `__init__`:

```python
self.clock = _ClockClass(self.root, w, h, initial_hour, initial_minute)
```

One flag in `constants.py` → one variable in `ui.py` → totally different clock.  
Both `Clock` and `DigitalClock` have the same public interface (`__init__`, `update_time`) — they are interchangeable.

**This is the key design decision in this file.** You'll edit the flag, not the wiring.

**What not to touch:**

`ui.py` manages the Tkinter event loop and all widget lifecycle.  
Bugs here can make the game unrunnable in hard-to-diagnose ways.

> Unless you're adding a brand-new UI component, leave `ui.py` alone.  
> Experiments belong in `gameplay/constants.py`, `gameplay/enums.py`, or the model files.

## Now: GameViewer

```
bwsi-sgai/
├── ui/
│   ├── ui.py
│   ├── game_viewer.py  <- now
│   ├── button_menu.py
│   ├── clock.py
│   ├── digital_clock.py
│   └── capacity_meter.py
```

*`GameViewer` is the central image panel — the largest widget in the window.*

*`bwsi-sgai/ui/game_viewer.py`*

## GameViewer — Locate + Orient

Open `ui/game_viewer.py`. It is 137 lines.

```python
class GameViewer:
    canvas: tk.Canvas   # the drawing surface
    photo: Any | None   # ImageTk.PhotoImage — held here to prevent garbage collection

    def __init__(self, root, w, h, data_fp, humanoid): ...
        # creates a Canvas at x=300, y=100, sized 50% of window width x 75% height
        # calls create_photo() to load the first humanoid image

    def create_photo(self, fp: str): ...
        # loads image from disk, resizes to canvas size, displays it

    def delete_photo(self, event=None): ...
        # removes the current image (called at game end, bound to <Delete> key)

    def display_score(self, score, end_reason=""): ...
        # overlays the final score on the canvas when the game ends
        # score dict: {'killed': int, 'saved': int, 'saved_value': float}

    # ... private helper ...
```

**You read from `GameViewer`; you don't edit it.** The humanoid image is loaded by `DataParser` and passed in as a path string.

In [1]:
import os
import sys
from pathlib import Path
%matplotlib inline

# Ensure the project root is the working directory and is on sys.path,
# so local packages (game_interfaces, gameplay, models) are importable
# and relative paths like DATA_DIR="data" resolve correctly.
# Works whether Jupyter was launched from bwsi-sgai/ or lectures/.
_here = Path().resolve()
_root = _here.parent if _here.name == "lectures" else _here
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [2]:
import inspect
from ui.game_viewer import GameViewer

# Read the class without opening a window
print(inspect.getsource(GameViewer.display_score))

    def display_score(
        self, score: dict[str, int | float], end_reason: str = ""
    ) -> None:
        """Render the final score over the canvas.

        Args:
            score: Dict with ``"killed"`` and ``"saved"`` integer counts and
                ``"saved_value"`` float (total value of saved humanoids),
                as returned by :meth:`~gameplay.scorekeeper.ScoreKeeper.get_score`.
            end_reason: Optional end-of-game reason string (e.g. ``"Time's Up"``
                or ``"The City Is Empty"``). Displayed as a bold header if provided.
        """
        if end_reason:
            tk.Label(
                self.canvas,
                text=end_reason,
                font=("Arial", 20, "bold"),
                bg=UI_COLORS["panel"],
                fg=UI_COLORS["accent"],
            ).pack(anchor=tk.NW)
        tk.Label(
            self.canvas,
            text="FINAL SCORE",
            font=("Arial", 30, "bold"),
            bg=UI_COLORS["panel"],
    

> `display_score()` reads `UI_COLORS["panel"]` and `UI_COLORS["accent"]` directly.  
> Change those colors in `constants.py` and the end-screen score panel changes too.

## Now: ButtonMenu

```
bwsi-sgai/
├── ui/
│   ├── ui.py
│   ├── game_viewer.py
│   ├── button_menu.py  <- now
│   ├── clock.py
│   ├── digital_clock.py
│   └── capacity_meter.py
```

*`ButtonMenu` renders the four action buttons and manages which ones are active.*

*`bwsi-sgai/ui/button_menu.py`*

## ButtonMenu — Locate + Orient

Open `ui/button_menu.py`. It is 144 lines.

```python
# At the top — button colors come from UI_COLORS in gameplay/constants.py:
_BTN_COLORS: dict[str, str] = {
    "Skip":   UI_COLORS["btn_skip"],
    "Squish": UI_COLORS["btn_squish"],
    "Save":   UI_COLORS["btn_save"],
    "Scram":  UI_COLORS["btn_scram"],
}

class ButtonMenu:
    canvas:  tk.Canvas            # contains the buttons
    buttons: dict[str, tk.Button] # keyed by base action name ("Save", etc.)

    def disable_all(self): ...
        # called at the top of every action callback to prevent double-click

    def disable_buttons(self, remaining_time, remaining_humanoids, at_capacity): ...
        # re-enables valid buttons after each turn
        # disables Save when ambulance is full
        # disables all when game is over
    # ... private helpers ...
```

**The interesting seam:** button colors are looked up from `UI_COLORS` at import time.  

In [3]:
from gameplay.constants import UI_COLORS
from gameplay.enums import ActionState

# ButtonMenu reads UI_COLORS for button backgrounds
print("Button colors from UI_COLORS:")
for key in ["btn_skip", "btn_squish", "btn_save", "btn_scram"]:
    print(f"  {key:15s} = {UI_COLORS[key]}")

print()
print("ActionState values (these become button labels):")
for action in ActionState:
    print(f"  ActionState.{action.name} = {action.value!r}")

Button colors from UI_COLORS:
  btn_skip        = #2e4057
  btn_squish      = #7b2d2d
  btn_save        = #1e4d35
  btn_scram       = #6b4c11

ActionState values (these become button labels):
  ActionState.SAVE = 'save'
  ActionState.SQUISH = 'squish'
  ActionState.SKIP = 'skip'
  ActionState.SCRAM = 'scram'


> Want to add a fifth button?  
> 1. Add a new value to `ActionState` in `gameplay/enums.py`  
> 2. Add a new `ActionCost` entry for it  
> 3. Add it to the `user_buttons` list in `ui/ui.py`  
> 4. Handle the new action in `ScoreKeeper`  
>
> Enums are the entry point, not the whole change.

## Now: Clock Components

```
bwsi-sgai/
├── ui/
│   ├── ui.py
│   ├── game_viewer.py
│   ├── button_menu.py
│   ├── clock.py         <- now
│   ├── digital_clock.py <- now
│   └── capacity_meter.py
```

*Two clocks, one flag. Which one runs is controlled by `LEGACY_CLOCK` in `gameplay/constants.py`.*

*`bwsi-sgai/ui/clock.py` · `bwsi-sgai/ui/digital_clock.py`*

## Clock Components — Locate + Orient

**`clock.py` — analog clock** (default when `LEGACY_CLOCK = True`)
```python
class Clock:
    def __init__(self, root, w, h, init_h, init_m): ...
        # draws a clock face from ui/graphics/clock.gif
        # hour hand color = UI_COLORS["accent"], minute hand = "#777777"
    def update_time(self, h: int, m: int): ...
        # redraws both hands to the new position
```

**`digital_clock.py` — digital countdown** (used when `LEGACY_CLOCK = False`)
```python
class DigitalClock:
    def __init__(self, root, w, h, init_h, init_m): ...
        # displays a tk.Label with bold text
    def update_time(self, h: int, m: int): ...
        # updates the label: "342 min\nremaining"
```

**Both share the same public interface** — `__init__(root, w, h, init_h, init_m)` and `update_time(h, m)`.  
That's what makes the feature-flag swap in `ui.py` work cleanly.

In [4]:
from gameplay.constants import LEGACY_CLOCK

print(f"LEGACY_CLOCK = {LEGACY_CLOCK}")
print()
if LEGACY_CLOCK:
    print("Active clock: Clock (analog)  -- ui/clock.py")
    print("To switch:    set LEGACY_CLOCK = False in gameplay/constants.py")
else:
    print("Active clock: DigitalClock    -- ui/digital_clock.py")
    print("To switch:    set LEGACY_CLOCK = True in gameplay/constants.py")

print()
print("Selection in ui.py:")
print("  _ClockClass = Clock if LEGACY_CLOCK else DigitalClock")

LEGACY_CLOCK = True

Active clock: Clock (analog)  -- ui/clock.py
To switch:    set LEGACY_CLOCK = False in gameplay/constants.py

Selection in ui.py:
  _ClockClass = Clock if LEGACY_CLOCK else DigitalClock


> This is a **feature flag** — a boolean constant that selects between two implementations.  
> It's a common pattern in production code for A/B testing or rolling out new features safely.  
> The key constraint: both implementations must have **identical method signatures**.
>
> Python docs: [Abstract Base Classes](https://docs.python.org/3/library/abc.html) — a formal way to enforce shared interfaces.

## Now: CapacityMeter

```
bwsi-sgai/
├── ui/
│   ├── ui.py
│   ├── game_viewer.py
│   ├── button_menu.py
│   ├── clock.py
│   ├── digital_clock.py
│   └── capacity_meter.py  <- now
```

*`CapacityMeter` shows how many ambulance slots are occupied as a grid of colored squares.*

*`bwsi-sgai/ui/capacity_meter.py`*

## CapacityMeter — Locate + Orient

Open `ui/capacity_meter.py`. It is 104 lines.

```python
class CapacityMeter:
    canvas:    tk.Canvas
    unit_size: int           # side length of each square (40px)
    _units:    list[int]     # canvas item IDs for each slot square

    def __init__(self, root, w, h, max_cap: int): ...
        # creates a canvas and draws max_cap squares in a grid
        # AMBULANCE_CAPACITY (from constants.py) is passed in as max_cap

    def update_fill(self, index: int): ...
        # clears all squares, then fills slots 0..index-1 in meter_filled color
        # called by ui.py after every action with scorekeeper.get_current_capacity()
    # ... private helpers ...
```

Color of filled slots = `UI_COLORS["meter_filled"]` (danger red `#e63946` by default).  
Number of slots = `AMBULANCE_CAPACITY` from `constants.py`.

In [5]:
from gameplay.constants import AMBULANCE_CAPACITY, UI_COLORS

print(f"Slots displayed:         {AMBULANCE_CAPACITY}")
print(f"Color when filled:       {UI_COLORS['meter_filled']}  (danger red)")
print(f"Color when empty:        {UI_COLORS['panel']}   (panel background)")
print()
print("To resize the ambulance: change AMBULANCE_CAPACITY in gameplay/constants.py")
print("To recolor filled slots:  change UI_COLORS['meter_filled'] in gameplay/constants.py")

Slots displayed:         10
Color when filled:       #e63946  (danger red)
Color when empty:        #242424   (panel background)

To resize the ambulance: change AMBULANCE_CAPACITY in gameplay/constants.py
To recolor filled slots:  change UI_COLORS['meter_filled'] in gameplay/constants.py


## Now: UI_COLORS

```
bwsi-sgai/
├── gameplay/
│   ├── constants.py  <- now (the customization file)
│   ├── humanoid.py
│   ├── scorekeeper.py
│   └── enums.py
├── ui/               <- reads from constants.py
├── models/
└── ...
```

*All color customization lives in `gameplay/constants.py`, not in `ui/`.  
Change colors there and every UI component picks them up automatically.*

*`bwsi-sgai/gameplay/constants.py`*

## UI_COLORS — Locate

Open `gameplay/constants.py` and scroll to the bottom. You'll find:

```python
UI_COLORS: dict[str, str] = {
    "bg":           "#1a1a1a",  # root and canvas background
    "panel":        "#242424",  # widget panels (slightly lighter)
    "fg":           "#f0f0f0",  # default text
    "btn_skip":     "#2e4057",  # muted blue  — neutral action
    "btn_squish":   "#7b2d2d",  # dark red    — destructive action
    "btn_save":     "#1e4d35",  # dark green  — positive action
    "btn_scram":    "#6b4c11",  # amber       — urgency / retreat
    "btn_default":  "#333333",  # machine mode / fallback buttons
    "btn_active":   "#4a4a4a",  # hover / active state
    "meter_filled": "#e63946",  # ambulance slot filled (danger red)
    "accent":       "#e63946",  # highlights and end-screen text
}
```

Every hex string is a standard web color code. Change any value, restart the game.  
No other files need to change — every UI component imports from here.

*`bwsi-sgai/gameplay/constants.py`*

## UI_COLORS — Orient: what each key controls

| Key | Where it appears | Default |
|-----|------------------|---------|
| `bg` | Root window background, all canvas backgrounds | `#1a1a1a` (near-black) |
| `panel` | Widget panel backgrounds (game viewer, clock, capacity) | `#242424` (dark gray) |
| `fg` | All text labels | `#f0f0f0` (near-white) |
| `btn_skip` | Skip button background | `#2e4057` (muted blue) |
| `btn_squish` | Squish button background | `#7b2d2d` (dark red) |
| `btn_save` | Save button background | `#1e4d35` (dark green) |
| `btn_scram` | Scram button background | `#6b4c11` (amber) |
| `btn_default` | Machine-mode advance button | `#333333` |
| `btn_active` | Button hover/pressed state | `#4a4a4a` |
| `meter_filled` | Filled ambulance slots | `#e63946` (danger red) |
| `accent` | End-screen headers, clock hour hand | `#e63946` |

In [6]:
from gameplay.constants import UI_COLORS, LEGACY_CLOCK

print("=== UI_COLORS ===")
for key, value in UI_COLORS.items():
    print(f"  {key:15s} = {value}")

print()
print(f"LEGACY_CLOCK = {LEGACY_CLOCK}")
clock_name = "Clock (analog)" if LEGACY_CLOCK else "DigitalClock"
print(f"Active clock:  {clock_name}")

=== UI_COLORS ===
  bg              = #1a1a1a
  panel           = #242424
  fg              = #f0f0f0
  btn_skip        = #2e4057
  btn_squish      = #7b2d2d
  btn_save        = #1e4d35
  btn_scram       = #6b4c11
  btn_default     = #333333
  btn_active      = #4a4a4a
  meter_filled    = #e63946
  accent          = #e63946

LEGACY_CLOCK = True
Active clock:  Clock (analog)


## Exercise: Change This and Run

**Time: under 5 minutes | Run in the terminal, not this notebook**

Tkinter opens a real OS window — the exercise runs from the terminal.

### Part 1: Change a button color

1. Open `gameplay/constants.py` in VS Code
2. Find `UI_COLORS` and change `"btn_save"` from `"#1e4d35"` to `"#5c1a5c"` (purple)
3. Save the file
4. Open a terminal at the repo root and run:
   ```bash
   python main.py -m human
   ```
5. **What to expect:** the Save button turns purple. All other colors stay the same.

### Part 2: Toggle the clock style

1. In `gameplay/constants.py`, find `LEGACY_CLOCK = True`
2. Change it to `LEGACY_CLOCK = False`
3. Save and relaunch: `python main.py -m human`
4. **What to expect:** the analog clock face disappears, replaced by a text label that says `"XXX min remaining"`.

Change it back to `True` when you're done.

In [7]:
# Preview: see your changes before relaunching
# Edit gameplay/constants.py, then run this cell to confirm the new value loaded.

# NOTE: if you changed constants.py, restart the kernel first (Kernel -> Restart Kernel)
# so the import below picks up the new values.

from gameplay.constants import UI_COLORS, LEGACY_CLOCK

print("Current btn_save color:", UI_COLORS["btn_save"])
print("Expected (after Part 1): #5c1a5c (purple)")
print()
print("Current LEGACY_CLOCK:   ", LEGACY_CLOCK)
print("Expected (after Part 2): False")

Current btn_save color: #1e4d35
Expected (after Part 1): #5c1a5c (purple)

Current LEGACY_CLOCK:    True
Expected (after Part 2): False


INSTRUCTOR NOTES:
Give students 3-4 minutes for Part 1. Confirm they found the right dict key.
Common mistake: editing ui/button_menu.py directly instead of constants.py. Redirect them.
Part 2 is fast — if they're on track, ask them to also try changing 'bg' to '#ffffff' (white) and observe the full contrast shift.
Remind students: after editing constants.py, they must relaunch the game (python main.py -m human) — the notebook cells need a kernel restart to pick up the change, but the game only needs a relaunch.

## Pop Quiz — Navigation

These are navigation questions. Which file do you open?

---

**Q1.** You want to make the Save button text appear in red instead of the default near-white.

A) `ui/button_menu.py`  
B) `gameplay/constants.py`  
C) `ui/ui.py`  
D) `gameplay/enums.py`

**Answer: B — `gameplay/constants.py`**  
Button text color comes from `UI_COLORS["fg"]`. Change that hex string — no touch to `button_menu.py`.  
The button *background* color is `UI_COLORS["btn_save"]`. Both live in the same dict.

## Pop Quiz — Navigation

**Q2.** You want to add a fifth button to the game called "Stabilize" with a new action type.

A) Only `ui/button_menu.py`  
B) `gameplay/enums.py` only  
C) `gameplay/enums.py` (new action + cost) + `ui/ui.py` (add to button list) + `gameplay/scorekeeper.py` (handle the action)  
D) `gameplay/constants.py` + `ui/ui.py`

**Answer: C**  
Enums are the entry point, not the whole change. You need:  
1. `gameplay/enums.py` — add `STABILIZE = "stabilize"` to `ActionState` and a cost to `ActionCost`  
2. `ui/ui.py` — add `("Stabilize", _on_stabilize)` to the `user_buttons` list  
3. `gameplay/scorekeeper.py` — add a `stabilize()` method  

`button_menu.py` handles it automatically — `ButtonMenu` gets whatever list `ui.py` passes in.

## Pop Quiz — Navigation

**Q3.** The analog clock face is showing but you want to switch to the digital countdown display. Which constant controls this, and where does it live?

A) `CLOCK_STYLE` in `ui/ui.py`  
B) `LEGACY_CLOCK` in `gameplay/constants.py`  
C) `USE_DIGITAL` in `ui/clock.py`  
D) `CLOCK_TYPE` in `ui/digital_clock.py`

**Answer: B — `LEGACY_CLOCK` in `gameplay/constants.py`**  
`LEGACY_CLOCK = True` → `Clock` (analog). `LEGACY_CLOCK = False` → `DigitalClock`.  
The selection happens in `ui/ui.py`: `_ClockClass = Clock if LEGACY_CLOCK else DigitalClock`.  
You edit the flag in `constants.py`; `ui.py` reads it. You never touch the clock files themselves.

## What You Now Know

- `ui/ui.py` — root window and wiring harness; **leave it alone** unless adding a new component
- `ui/game_viewer.py` — humanoid image panel; **read from it**, don't edit in normal use
- `ui/button_menu.py` — four action buttons; **edit here** to add/remove buttons (plus enums + scorekeeper)
- `ui/clock.py` + `ui/digital_clock.py` — two interchangeable clock widgets; **toggle with `LEGACY_CLOCK`**
- `ui/capacity_meter.py` — ambulance slot grid; **read from it**, driven by `ScoreKeeper`
- `gameplay/constants.py` → `UI_COLORS` — **all color customization lives here**, not in `ui/`

The core pattern:
```
gameplay/constants.py  (you edit this)
        |__ imported by every UI file automatically
```

Change a color → restart the game. Done.

---

**Next: Module 4 — Architecture & Wiring**  
How does `main.py` start the game? How does `base_env.py` connect the data, models, and UI into one game loop?  
We'll trace a full `auto` game turn end-to-end through `main.py` → `base_env.py` → models.